# 02 · Preprocessing dan normalisasi

**Analisis awal, bukan solusi final.** Notebook ini membereskan masalah yang ditemukan di
`01_eda`, lalu menyimpan tabel bersih untuk notebook berikutnya.

Data mentah hasil scrape **tidak diubah**. Semua perapian terjadi di sini, pada salinan,
dan hasilnya disimpan di `data/process/<run>/tables/` (folder `data/` tidak masuk git).

| Langkah | Masalah dari EDA | Yang dilakukan |
|---|---|---|
| 1 | Nama brand tidak seragam | Satu kunci per brand, alias untuk sub-lini, satu ejaan tampilan |
| 2 | Label sendiri dan vendor pengisi | Ditandai, tidak dihitung sebagai brand |
| 3 | Toko punya banyak kolom mentah | Tabel toko ringkas: pelanggan atau bukan, lokasi, teritori |
| 4 | Harga campur mata uang dan ada nilai ekstrem | Hanya USD yang dibandingkan, nilai tak wajar ditandai |
| 5 | Bahan dan kategori sebagian kosong | Dikelompokkan, yang kosong tetap "tidak diketahui" |

In [ ]:
import nbutil as nb
import pandas as pd

print("Run:", nb.run_id())
products = nb.load_products()
stores = nb.load_stores()
knit = nb.load_knit()

## 1. Normalisasi nama brand

Kunci brand memakai fungsi yang sama dengan sistem (`compact`): huruf kecil, tanpa spasi
dan tanda baca. Jadi "White + Warren", "White & Warren" dan "WHITE+WARREN" menjadi satu.

Yang tidak tertangkap oleh aturan itu ditulis di tabel alias. Tabel ini masih pendek dan
diisi dengan tangan: nanti sebaiknya dipindah ke `brands.csv`.

In [ ]:
ALIASES = {
    "zsupplylounge": "zsupply",
    "zsupplyactivewear": "zsupply",
    "whiteandwarren": "whitewarren",
    "lilap": "lillap",
    "railsclothing": "rails",
}
PLACEHOLDERS = {
    "",
    "default",
    "vendor",
    "na",
    "none",
    "unknown",
    "nobrand",
    "generic",
    "misc",
    "other",
}


def brand_key(vendor: object) -> str:
    key = nb.compact(vendor) if isinstance(vendor, str) else ""
    return ALIASES.get(key, key)


products["brand_key"] = products["vendor"].map(brand_key)
before = products["vendor"].fillna("").str.strip().replace("", pd.NA).nunique()
print(f"Nama vendor mentah: {before:,}")
print(f"Kunci brand:        {products.loc[products['brand_key'].ne(''), 'brand_key'].nunique():,}")

Ejaan yang ditampilkan untuk tiap brand adalah ejaan yang paling sering dipakai toko.

In [ ]:
named = products[products["brand_key"].ne("")]
display_name = named.groupby("brand_key")["vendor"].agg(
    lambda s: s.str.strip().value_counts().index[0]
)
spellings = named.groupby("brand_key")["vendor"].agg(lambda s: sorted(set(s.str.strip())))
brands_dim = pd.DataFrame({"brand": display_name, "spellings": spellings.map(len)}).reset_index()
brands_dim.sort_values("spellings", ascending=False).head(8)

## 2. Menandai label sendiri dan vendor pengisi

Vendor yang sama dengan nama toko berarti toko itu menjual labelnya sendiri. Itu bukan
brand yang "dipasok ke toko", jadi tidak masuk peta brand dan toko.

In [ ]:
products["own_label"] = products["brand_key"] == products["domain"].map(nb.domain_core)
products["placeholder"] = products["brand_key"].isin(PLACEHOLDERS)
products["knit"] = products["knit_kind"].fillna("").ne("")
kept = products[~products["own_label"] & ~products["placeholder"]]
print(f"Produk total:              {len(products):,}")
print(f"  label sendiri:           {int(products['own_label'].sum()):,}")
print(f"  vendor kosong/pengisi:   {int(products['placeholder'].sum()):,}")
print(f"  dipakai untuk peta brand: {len(kept):,}")

## 3. Peta brand dan toko

Ini tabel inti: satu baris per pasangan brand dan toko. Dibangun dari tabel produk lengkap,
bukan dari kolom `brands` di sheet analisis yang hanya memuat 15 brand per toko.

In [ ]:
brand_stores = (
    kept.groupby(["brand_key", "domain"])
    .agg(products=("url", "size"), knit_products=("knit", "sum"))
    .reset_index()
    .merge(brands_dim[["brand_key", "brand"]], on="brand_key")
)
brand_stores["source"] = "vendor"
print(f"{len(brand_stores):,} pasangan brand dan toko")
print(f"{brand_stores['brand_key'].nunique():,} brand · {brand_stores['domain'].nunique()} toko")

check = brand_stores[
    brand_stores["brand_key"].isin(["zsupply", "rails", "whitewarren", "lillap", "woodenships"])
]
check.groupby("brand").agg(
    toko=("domain", "nunique"), produk=("products", "sum"), knitwear=("knit_products", "sum")
)

## 4. Tabel toko yang ringkas

Tiga hal yang dibutuhkan analisis dari tiap toko: apakah terbaca, apakah sudah pelanggan
kita, dan di mana lokasinya terhadap stockist kita.

`is_customer` saat ini hanya berdasarkan daftar stockist dan produk Wooden Ships yang
terlihat di katalog. Akun Salesforce belum ikut, jadi sebagian "bukan pelanggan" bisa keliru.

In [ ]:
TERRITORY_MILES = 15

stores_clean = stores[
    ["domain", "store_name", "status", "segment", "score", "store_type", "business_type", "platform",
     "city", "state", "country", "lat", "lng", "nearest_stockist_miles", "knit_products", "brand_count",
     "knit_price_median", "currency"]
].copy()  # fmt: skip
stores_clean["readable"] = stores["status"].eq("ok")
stores_clean["is_customer"] = stores["segment"].eq("existing_customer") | stores[
    "carries_wooden_ships"
].astype(str).str.lower().eq("true")
stores_clean["has_email"] = stores["emails"].map(nb.json_list).map(len).gt(0)
stores_clean["state"] = stores_clean["state"].str.upper().str.strip()
stores_clean["territory"] = "lokasi tidak diketahui"
known = stores_clean["nearest_stockist_miles"].notna()
stores_clean.loc[
    known & (stores_clean["nearest_stockist_miles"] <= TERRITORY_MILES), "territory"
] = "ada stockist dalam radius"
stores_clean.loc[
    known & (stores_clean["nearest_stockist_miles"] > TERRITORY_MILES), "territory"
] = "wilayah kosong"

print(stores_clean.loc[stores_clean["readable"], "territory"].value_counts().to_string())
print(
    f"\nPelanggan kita di antara toko terbaca: {int(stores_clean.loc[stores_clean['readable'], 'is_customer'].sum())}"
)

## 5. Normalisasi harga

Aturan proyek: harga tidak dikonversi. Jadi perbandingan harga hanya memakai produk yang
memang ditulis dalam USD. Nilai di luar rentang wajar ditandai, bukan dihapus.

In [ ]:
PRICE_MIN, PRICE_MAX = 5, 2000

knit_clean = knit[
    ["domain", "title", "vendor", "knit_kind", "category", "main_material", "materials", "price", "currency",
     "on_sale", "product_type", "url"]
].copy()  # fmt: skip
knit_clean["brand_key"] = knit_clean["vendor"].map(brand_key)
knit_clean["usd"] = knit_clean["currency"].eq("USD")
knit_clean["price_ok"] = knit_clean["price"].between(PRICE_MIN, PRICE_MAX)
knit_clean["price_usd"] = knit_clean["price"].where(knit_clean["usd"] & knit_clean["price_ok"])

print(f"Knitwear total:               {len(knit_clean):,}")
print(f"  ditulis dalam USD:          {int(knit_clean['usd'].sum()):,}")
print(
    f"  harga di luar ${PRICE_MIN} sampai ${PRICE_MAX:,}: {int((~knit_clean['price_ok']).sum()):,}"
)
print(f"  dipakai untuk banding harga: {int(knit_clean['price_usd'].notna().sum()):,}")

## 6. Normalisasi kategori dan bahan

Kategori sudah seragam dari tahap analisis. Bahan dikelompokkan menjadi keluarga serat,
supaya "merino" dan "wool" dianggap serupa. Yang kosong tetap kosong: **tidak diketahui
bukan berarti tidak serupa.**

In [ ]:
FIBRE_FAMILY = {
    "cotton": "katun", "linen": "katun",
    "wool": "wol", "merino": "wol", "yak": "wol", "camel": "wol",
    "cashmere": "serat halus", "alpaca": "serat halus", "mohair": "serat halus", "silk": "serat halus",
    "synthetic": "sintetis", "viscose": "sintetis",
}  # fmt: skip

knit_clean["material_family"] = knit_clean["main_material"].map(FIBRE_FAMILY)
knit_clean["materials"] = knit_clean["materials"].map(nb.json_list)
pd.crosstab(knit_clean["material_family"].fillna("(tidak diketahui)"), knit_clean["knit_kind"])

## 7. Simpan tabel bersih

In [ ]:
nb.save(brand_stores, "brand_stores")
nb.save(brands_dim, "brands_dim")
nb.save(stores_clean, "stores_clean")
nb.save(knit_clean, "knit_clean")
text = kept.loc[kept["knit"], ["brand_key", "domain", "title", "product_type", "tags"]].copy()
text["tags"] = text["tags"].map(lambda v: " ".join(v) if v is not None else "")
nb.save(text, "knit_text")

for name in ("brand_stores", "brands_dim", "stores_clean", "knit_clean", "knit_text"):
    print(f"{name:14} {len(nb.load(name)):>8,} baris")
print("\nDisimpan di:", nb.out_dir().relative_to(nb.ROOT))

## Yang belum dibereskan di sini

- **Tabel alias masih pendek.** Sub-lini dan ejaan dengan kata "and" hanya tertangkap bila
  ditulis di `ALIASES`.
- **Toko yang sama dengan dua domain** belum digabung.
- **Mata uang selain USD** tidak ikut perbandingan harga, termasuk toko Kanada.
- **Lokasi** hanya ada untuk toko yang menulis alamat di situsnya.